# 08 — Reactive Basics

`data_setter`, `data_formula`, `subscribe()` — the reactive engine.

**What you learn:**
- `data_setter`: static value in the reactive store
- `data_formula`: computed value, re-executes when deps change
- `ReactiveManager` with `subscribe=True`
- Formula dependency chains via topological sort

**Prerequisites:** 04_pointers

In [ ]:
from pathlib import Path
from IPython.display import HTML

from genro_bag import Bag
from genro_bag.resolvers import FileResolver
from genro_builders.contrib.html import HtmlManager
from genro_toolbox import set_sync

set_sync()

In [ ]:
class PriceCalculator(HtmlManager):
    def main(self, source):
        source.data_setter('base_price', value='^base_price')
        source.data_setter('discount', value='^discount')
        source.data_setter('tax_rate', value='^tax_rate')
        source.data_formula('net_price',
            func=lambda base_price, discount: base_price * (1 - discount),
            base_price='^base_price', discount='^discount')
        source.data_formula('total',
            func=lambda net_price, tax_rate: round(net_price * (1 + tax_rate), 2),
            net_price='^net_price', tax_rate='^tax_rate')
        # FileResolver loads style.css lazily at render time (pull model)
        source.head().style(FileResolver('style.css'))
        body = source.body()
        body.h1('Price Calculator')
        body.p('^base_price')
        body.p('^net_price', _class='result')
        body.p('^total', _class='result')

In [ ]:
app = PriceCalculator()
app.local_store('page').fill_from(
    Bag.from_json(Path('data.json').read_text()),
)
app.run(subscribe=True)
store = app.local_store('page')
print(f'base={store["base_price"]}, net={store["net_price"]}, total={store["total"]}')
HTML(app.render())

## Change data — formulas re-execute automatically

In [ ]:
store['base_price'] = 200
print(f'base={store["base_price"]}, net={store["net_price"]}, total={store["total"]}')
HTML(app.render())

In [ ]:
store['discount'] = 0.25
print(f'base={store["base_price"]}, net={store["net_price"]}, total={store["total"]}')
HTML(app.render())